In [2]:
import numpy as np
import pandas as pd
import numba
import math

import xarray as xr
import cf_xarray
import glob
import os

from xgcm import Grid
from xmitgcm import open_mdsdataset
import ecco_v4_py as ecco
import gsw

import cmocean

import matplotlib.pyplot as plt
import matplotlib as mpl
# %matplotlib inline


import warnings
warnings.simplefilter("ignore") 

YEAR_NUMBERS = np.arange(51,66)


lat_range = np.arange(35., 71., 1.) # lat values at which overturning is computed; I double-checked so it's fully covered by new resampled files 

file_in_suffix = ".nc"
file_out_suffix = ".nc" #str(int(lat_range[0])) + "_"+ str(int(lat_range[-1])) +"N.nc"

EXPERIMENT_IN = 'CONTROL_ABS_WINDS_NO_SUBMESO'
EXPERIMENT_OUT = 'CONTROL_NEW/'

PATH_data = '/gws/nopw/j04/snapdragon/markina/DIAGNOSTICS/'+EXPERIMENT_IN+'/MONTHLY_outputs/'

PATH_grid = '/gws/nopw/j04/snapdragon/markina/GRID/'
PATH_plots = '/home/users/markina/Plots/'

# PATH_out = '/gws/nopw/j04/snapdragon/markina/DIAGNOSTICS/SENSITIVITY_TESTS/'+EXPERIMENT+'/MOC_sigma_basin/'
PATH_out = '/home/users/markina/Data/'+ EXPERIMENT_OUT + 'MOC_sigma_basin_from_monthly_means/'

sigma_0_min=25. # CHANGE this later (if needed)
sigma_0_max=28.5

delta_sigma_0 = 0.05

target_sigma_0_levels = np.linspace(sigma_0_min, sigma_0_max, int((sigma_0_max-sigma_0_min)/delta_sigma_0 + 1))



# Hemant's functions (inspired by ECCO)
def transport_z(ds_vel, z_lev, grid, assign_name='transport'):

    """Compute volume transport per unit horizontal length
    Parameters
    ----------
    ds_vel : xarray DataArray - velocity data
    z_lev : xarray DataArray - outer z levels
    grid : xgcm Grid object
    assign_name : name for the volume transport

    Returns
    -------
    transport : xarray DataArray for volume transport
    """

    thickness = grid.diff(z_lev, 'Z') # compute vertical grid spacing
    transport =  ds_vel * thickness # velocity x vertical grid spacing
    transport = transport.fillna(0.).rename(assign_name)

    return transport

def transport_sigma(pot_density, transport, density_levels, grid, dim=None, method='linear'): 

    """Compute volume transport in density layers using xgcm.transform method
    Parameters
    ----------
    pot_density : xarray DataArray - potential density
    transport : xarray DataArray - volume transport on z-levels
    density_levels : numpy array - Target density levels at which one wants to compute volume transport
    grid : xgcm Grid object
    dim : dimension for interpolation to have ensure pot_density and transport on the same grid
    method : transform method out of 'linear' or 'conservative'

    Returns
    -------
    transport_sigma : xarray DataArray - volume transport at denisty levels
    """

    if(dim != None): # interpolate the density to transport data grid
        sigma_interp = grid.interp(pot_density, [dim], boundary='extend')
    else:
        sigma_interp = pot_density

    sigma_interp = sigma_interp.rename('sigma0')
    transport_sigma = grid.transform(transport, 'Z', density_levels,
                                     target_data=sigma_interp, method=method)
    transport_sigma.sigma0.attrs['units'] = "kg/m^3"
    transport_sigma.sigma0.attrs['long_name'] = "Potential density with reference to ocean surface - 1000."

    return transport_sigma


def meridional_overturning(ds_var, transport_u, transport_v, latitude, lat_vals,
                           grid, dx, dy, dim='z'):

    """Compute meridional overturning at specified latitudes
    Parameters
    ----------
    ds_var : xarray Dataarray - To save overturning data
    transport_u : xarray DataArray - Zonal transport
    transport_v : xarray DataArray - Meridional transport
    latitude : xarray DataArray - latitude values at cell centers
    lat_vals : Numpy array for range of lattiudes
    grid : xgcm Grid object
    dx : xarray DataArray - Zonal grid spacing at meridional velocity grid
    dy : xarray DataArray - Meridional grid spacing at zonal velocity grid
    mask_u : xarray Dataset - u mask for the Region of interest
    mask_v : xarray Dataset - v mask for the Region of interest
    dim : vertical dimension name
    Returns
    -------
    ds_var : xarray DataArray - meridional overturning at lat_vals
    """

    for lat in lat_vals:
        
        if lat%5==0:
            print(lat)

        lat_maskW, lat_maskS = get_latitude_masks(lat, latitude, grid)

#         print('computing x transport')
        lat_trsp_x = ((transport_u * lat_maskW * dy)).sum(dim=['i_g','j'])
#         print('computing y transport')
        lat_trsp_y = ((transport_v * lat_maskS * dx)).sum(dim=['i','j_g'])
#         print('writing to output variable')
        ds_var.loc[{'lat':lat}] = lat_trsp_x + lat_trsp_y
#         print('done')
    ds_var = ds_var.cumsum(dim=dim)

    return ds_var


def get_latitude_masks(lat_val, yc, grid): # adopted from ecco_v4_py

    """Compute maskW/S which grabs vector field grid cells along specified latitude band
    Parameters
    ----------
    lat_val : float - latitude at which to compute mask
    yc : xarray DataArray - Contains latitude values at cell centers
    grid : xgcm Grid object
    Returns
    -------
    maskWedge, maskSedge : xarray DataArray
        contains masks of latitude band at grid cell west and south grid edges
    """
    # Compute difference in X, Y direction.
    # multiply by 1 so that "True" -> 1, 2nd arg to "where" puts False -> 0
    ones = xr.ones_like(yc)
    maskC = ones.where(yc >= lat_val, 0)

    maskWedge = grid.diff(maskC, ['X'], boundary='fill')
    maskSedge = grid.diff(maskC, ['Y'], boundary='fill')

    return maskWedge, maskSedge



jj_min = 700  # these are the same indices as used in state 2d fields
jj_max = 1400

ii_min = 250
ii_max = 1400 

# jj_min = 400  # corresponds to input files
# jj_max = 900

# ii_min = 0
# ii_max = 1200 

# jj_min = 0  # corresponds to files with tropics
# jj_max = 700

# ii_min = 0
# ii_max = 1350


x_smol = np.arange(ii_min,ii_max)
y_smol = np.arange(jj_min,jj_max) 



nx=1440
ny=1440
nz=50

# x = np.arange(0,nx)
# y = np.arange(0,ny)




YC_filename = PATH_grid + 'YC.data'
YC_tmp = np.fromfile(YC_filename, dtype='>f') 
YC = YC_tmp.reshape(ny,nx)

XC_filename = PATH_grid + 'XC.data'
XC_tmp = np.fromfile(XC_filename, dtype='>f') 
XC = XC_tmp.reshape(ny,nx)

da_lat = xr.DataArray(data=YC[jj_min:jj_max, ii_min:ii_max], dims=["j", "i"])
da_lon = xr.DataArray(data=XC[jj_min:jj_max, ii_min:ii_max], dims=["j", "i"])



dz = [10.00, 10.00, 10.00, 10.00, 10.00, 10.00, 10.00, 10.01,10.03,\
      10.11, 10.32, 10.80, 11.76, 13.42, 16.04, 19.82, 24.85, \
      31.10, 38.42, 46.50, 55.00, 63.50, 71.58, 78.90, 85.15, 90.18, \
      93.96, 96.58, 98.25, 99.25,100.01,101.33,104.56,111.33,122.83, \
      139.09,158.94,180.83,203.55,226.50,249.50,272.50,295.50,318.50, \
      341.50,364.50,387.50,410.50,433.50,456.50]


Zp1 = np.zeros(len(dz)+1)

Zp1[0] = 0.

for idepth in range(1,len(dz)+1):
    Zp1[idepth] = Zp1[idepth-1]+dz[idepth-1]

Zp1 = -1.*Zp1


Z = np.zeros(len(dz))

for idepth in range(0,len(dz)):
    if idepth==0:
        Z[idepth] = (Zp1[idepth+1]-Zp1[idepth])*0.5 + 0.
    else:
        Z[idepth] = (Zp1[idepth+1]-Zp1[idepth])*0.5 + Zp1[idepth]


YC_new = np.reshape(YC, (1,ny,nx))

drF_filename = PATH_grid + 'DRF.data'
drF_tmp = np.fromfile(drF_filename, dtype='>f') # >f correponds to big endian binary file (that is used by MITgcm)
drF = drF_tmp.reshape(nz) 

dyG_filename = PATH_grid + 'DYG.data'
dyG_tmp = np.fromfile(dyG_filename, dtype='>f')
dyG = dyG_tmp.reshape(ny,nx) 
# dyG = dyG_src[jj_section,ii_min:ii_max]
# del(dyG_src)

# plt.contourf(dyG) # should be a nice smooth plot

dxG_filename = PATH_grid + 'DXG.data'
dxG_tmp = np.fromfile(dxG_filename, dtype='>f') 
dxG = dxG_tmp.reshape(ny,nx) 
# dxG = dxG_src[jj_section,ii_min:ii_max]
# del(dxG_src)

    
# Read variables relevant for computation of density

#Reading hydrostatic pressure and reference density - needed for computing density field to define density surfaces 

PHrefC_filename = PATH_grid + 'PHrefC.data' 
PHrefC_tmp = np.fromfile(PHrefC_filename, dtype='>f') # >f correponds to big endian binary file (that is used by MITgcm)
pressure_m2_s2 = PHrefC_tmp.reshape(nz) # Reference Hydrostatic Pressure [m2 s-2]
# plt.plot(pressure_m2_s2) # should be a nice smooth plot

RhoRef_filename = PATH_grid + 'RhoRef.data'
RhoRef_tmp = np.fromfile(RhoRef_filename, dtype='>f') # >f correponds to big endian binary file (that is used by MITgcm)
density = RhoRef_tmp.reshape(nz) 
pressure_dbar = pressure_m2_s2*density/10000. # 1 dbar = 10 KPa = 10000 Pa

da_pres = xr.DataArray(data=pressure_dbar, dims=["k"],)
da_z = xr.DataArray(data=Z, dims=["k"])
da_zp1 = xr.DataArray(data=Zp1, dims=["k"])

da_dxg = xr.DataArray(data=dxG[jj_min:jj_max, ii_min:ii_max], dims=['j_g', 'i'])
da_dyg = xr.DataArray(data=dyG[jj_min:jj_max, ii_min:ii_max], dims=['j', 'i_g'])


In [3]:
months = ['05', '06', '07', '08', '09', '10', '11', '12', '01', '02', '03', '04']

In [ ]:
%%time

for YEAR in YEAR_NUMBERS:

    # YEAR = 60 # change this later - this is just for test

    for imonth in np.arange(12):

        print(str(YEAR) + '/' + str(months[imonth])+ " in progress...")

        print('reading data (state)...')

        ds_temp = xr.open_dataset(PATH_data+str(YEAR)+"_state_3d_temp_monthly" + file_in_suffix).isel(time=imonth)
        ds_sal = xr.open_dataset(PATH_data+str(YEAR)+"_state_3d_sal_monthly" + file_in_suffix).isel(time=imonth)


        print('reading data (trsp)...')

        ds_uvel = xr.open_dataset(PATH_data+str(YEAR)+"_trsp_3d_uvel_monthly" + file_in_suffix).isel(time=imonth)
        ds_vvel = xr.open_dataset(PATH_data+str(YEAR)+"_trsp_3d_vvel_monthly" + file_in_suffix).isel(time=imonth)

        ds = xr.Dataset(data_vars=dict(THETA=(['k','j', 'i'], ds_temp.temp.values),
                                       SALT=(['k','j', 'i'], ds_sal.sal.values),
                                       UVELMASS=(['k','j', 'i_g'], ds_uvel.uvel.values),
                                       VVELMASS=(['k','j_g', 'i'], ds_vvel.vvel.values),),
                         coords=dict(k=(["k"], np.arange(0, nz)),
                                     j=(["j"], y_smol),
                                     j_g=(["j_g"], y_smol),
                                     i_g=(["i_g"], x_smol),
                                     i=(["i"], x_smol),) )

        del ds_temp, ds_sal, ds_uvel, ds_vvel

        ds['pressure_dbar']=da_pres
        ds['Z']=da_z
        ds['DXG']=da_dxg
        ds['DYG']=da_dyg
        # ds['Z']=da_z

        ds['lat']=da_lat
        ds['lon']=da_lon

        print('computing density...')

        potential_temp = ds.THETA
        salin = ds.SALT 


        # ds['pressure_dbar']=da_pres
        # ds['lat']=da_lat
        # ds['lon']=da_lon

        absolute_salin = gsw.SA_from_SP(salin, ds.pressure_dbar - 10.1325, ds.lon, ds.lat)
        conservative_temp = gsw.CT_from_pt(absolute_salin, potential_temp)
        sigma_2_tmp = gsw.sigma2(absolute_salin, conservative_temp)
        sigma_0_tmp = gsw.sigma0(absolute_salin, conservative_temp)

        sigma_2 = sigma_2_tmp.to_dataset(name='SIGMA_2')
        sigma_0 = sigma_0_tmp.to_dataset(name='SIGMA_0')

        ds['SIGMA_0'] = sigma_0.SIGMA_0
        ds['SIGMA_2'] = sigma_2.SIGMA_2

        del absolute_salin, conservative_temp, sigma_2_tmp, sigma_0_tmp, sigma_0, sigma_2


        level_outer_data = Zp1
        level_outer = xr.DataArray(
            level_outer_data,
            dims=['level_outer'],
            coords={'level_outer': ('level_outer', level_outer_data)}
        )
        ds = ds.assign_coords({'level_outer': level_outer})


        grid = Grid(ds, coords={'Z': {'center': 'k', 'outer': 'level_outer'},
                                'X': {'center': 'i', 'left': 'i_g'},
                                'Y': {'center': 'j', 'left': 'j_g'}
                                },
                    periodic=False,
                    )

        print('computing transport(z)')

        # compute zonal and meridional transport
        Zonal_Transport = transport_z(ds['UVELMASS'], ds['level_outer'], grid,
                                      assign_name='Zonal_Transport')

        Meridional_Transport = transport_z(ds['VVELMASS'], ds['level_outer'], grid,
                                           assign_name='Meridional_Transport')

        print('computing transport(sigma)')
        # compute zonal/meridional transport on sigma levels and depth of density isolines
        Zonal_Transport_sigma = transport_sigma(ds.SIGMA_0, Zonal_Transport, target_sigma_0_levels,
                                                grid=grid, dim='X', method='conservative')

        del Zonal_Transport

    #     print('computing meridional transport(sigma)')

        Meridional_Transport_sigma = transport_sigma(ds.SIGMA_0, Meridional_Transport, target_sigma_0_levels,
                                                     grid=grid, dim='Y',method='conservative')

        del Meridional_Transport

        ds_save = xr.Dataset() # creating dataset for saving results

        # first create xarray dataarray for overturning vars
        tmp_array_sigma = np.empty((len(Meridional_Transport_sigma['sigma0']), len(lat_range)))
        ds_save['Overturning_sigma'] = xr.DataArray(data=tmp_array_sigma.copy(),
                                                    coords={'sigma0': Zonal_Transport_sigma['sigma0'],
                                                            'lat': lat_range},
                                                    dims=[ 'sigma0','lat'])


        ds_save['Overturning_sigma'] = meridional_overturning(ds_save['Overturning_sigma'],
                                                          Zonal_Transport_sigma, Meridional_Transport_sigma, ds['lat'], lat_range,
                                                          grid, ds['DXG'], ds['DYG'], dim='sigma0')


        ds_save.to_netcdf(PATH_out+"MOC_sigma_0_yr_"+str(YEAR)+"_month_"+str(months[imonth])+"_with_xgcm" + file_out_suffix)

        del grid,ds

        print('done!') 

51/05 in progress...
reading data (state)...
reading data (trsp)...
computing density...
computing transport(z)
computing transport(sigma)
35.0
40.0
45.0
50.0
55.0
60.0
65.0
70.0
done!
51/06 in progress...
reading data (state)...
reading data (trsp)...
computing density...
computing transport(z)
computing transport(sigma)
35.0
40.0
45.0
50.0
55.0
60.0
65.0
70.0
done!
51/07 in progress...
reading data (state)...
reading data (trsp)...
computing density...
computing transport(z)
computing transport(sigma)
35.0
40.0
45.0
50.0
55.0
60.0
65.0
70.0
done!
51/08 in progress...
reading data (state)...
reading data (trsp)...
computing density...
computing transport(z)
computing transport(sigma)
35.0
40.0
45.0
50.0
55.0
60.0
65.0
70.0
done!
51/09 in progress...
reading data (state)...
reading data (trsp)...
computing density...
computing transport(z)
computing transport(sigma)
35.0
40.0
45.0
50.0
55.0
60.0
65.0
70.0
done!
51/10 in progress...
reading data (state)...
reading data (trsp)...
computi